# ch07 Bonus 01：直接偏好优化（DPO）

> 对照官方 `ch07/04_preference-tuning-with-dpo`

## 一句话

让模型在「好回答 vs 坏回答」之间学习偏好，**无需训练奖励模型**，直接优化策略——这是 RLHF 的简化替代。

## 背景：RLHF 太复杂

传统 RLHF（如 ChatGPT 初版）三步走：
1. 训练奖励模型（reward model）
2. 用强化学习（PPO）优化策略
3. 采样、打分、迭代……

DPO 的洞察：**可以跳过奖励模型和 RL**，直接用偏好数据（chosen vs rejected）通过一个简洁的损失函数优化策略。

## DPO 损失

$$\mathcal{L}_{DPO} = -\log\sigma\left(\beta\left[\log\frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)}\right]\right)$$

- $y_w$ = chosen（好回答），$y_l$ = rejected（坏回答）
- $\pi$ = 当前策略，$\pi_{ref}$ = 冻结的参考策略
- $\beta$ = 温度，控制偏离参考模型的程度

> 直觉：让「好回答」的相对对数概率（相对参考模型）升上去，「坏回答」的降下来。

In [ ]:
import torch
import torch.nn.functional as F
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M


def compute_log_probs(model, sequences, targets):
    """计算序列的对数概率（DPO 需要比较回答的好坏概率）。"""
    logits = model(sequences)
    log_probs = F.log_softmax(logits, dim=-1)
    # 用 clamp 处理 -100 ignore index（gather 不能用负索引）
    safe_targets = targets.clamp(min=0)
    selected = log_probs.gather(-1, safe_targets.unsqueeze(-1)).squeeze(-1)
    mask = (targets != -100).float()
    return (selected * mask).sum(dim=-1)


def dpo_loss(policy_chosen_lp, policy_rejected_lp,
             ref_chosen_lp, ref_rejected_lp, beta=0.1):
    """DPO 损失：让 chosen 的相对概率高于 rejected。"""
    log_ratio_chosen = policy_chosen_lp - ref_chosen_lp
    log_ratio_rejected = policy_rejected_lp - ref_rejected_lp
    logits = beta * (log_ratio_chosen - log_ratio_rejected)
    return -F.logsigmoid(logits).mean()

In [ ]:
# 构造偏好数据：同一 prompt，一个好回答一个坏回答
tok = tiktoken.get_encoding("gpt2")

def make_seq(prompt, response, max_len=64):
    """构造 (input, target) 对，prompt 部分 mask。"""
    p_ids = tok.encode(prompt)
    r_ids = tok.encode(response)
    ids = p_ids + r_ids
    targets = ids[1:] + [tok.eot_token]
    for i in range(len(p_ids)):
        if i < len(targets):
            targets[i] = -100
    ids = ids[:max_len] + [50256]*(max_len - min(len(ids), max_len))
    targets = targets[:max_len] + [-100]*(max_len - min(len(targets), max_len))
    return torch.tensor([ids]), torch.tensor([targets])

prompt = "### Instruction:\n翻译成英文\n### Input:\n你好\n### Response:\n"
chosen_resp = " Hello"      # 好回答（正确翻译）
rejected_resp = " Hola"     # 坏回答（西语，题目要英文）

ch_x, ch_y = make_seq(prompt, chosen_resp)
rj_x, rj_y = make_seq(prompt, rejected_resp)
print(f"偏好数据: chosen={chosen_resp!r}, rejected={rejected_resp!r}")
print(f"chosen 序列长度: {ch_x.shape[1]}, 计算 loss 位置: {(ch_y[0]!=-100).sum().item()}")

In [ ]:
# DPO 训练：policy 在偏好数据上优化，ref 冻结
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 64})
torch.manual_seed(123)
policy = GPTModel(cfg)        # 待优化的策略
ref = GPTModel(cfg)           # 冻结的参考模型（初始与 policy 相同）
ref.eval()
for p in ref.parameters():
    p.requires_grad = False

# 参考模型的对数概率（只需算一次，detach 不参与梯度）
with torch.no_grad():
    ref_chosen_lp = compute_log_probs(ref, ch_x, ch_y)
    ref_rejected_lp = compute_log_probs(ref, rj_x, rj_y)

optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-3)

print("\nDPO 优化过程（让模型偏好 Hello 而非 Hola）：")
print(f"{'step':<6} {'loss':<10} {'chosen-rejected margin':<25}")
for step in range(30):
    optimizer.zero_grad()
    pol_chosen = compute_log_probs(policy, ch_x, ch_y)
    pol_rejected = compute_log_probs(policy, rj_x, rj_y)
    loss = dpo_loss(pol_chosen, pol_rejected, ref_chosen_lp, ref_rejected_lp, beta=0.5)
    loss.backward()
    optimizer.step()
    margin = (pol_chosen.item() - pol_rejected.item())
    if step % 6 == 0 or step == 29:
        print(f"{step:<6} {loss.item():<10.4f} {margin:+.3f}")

print("\n✓ chosen-rejected margin 持续上升，说明模型学会偏好 Hello（好回答）。")
print("  DPO 无需奖励模型和 RL，直接用偏好数据优化策略。")